In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Konfigurasi mitigasi error

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



> **Note:** Rilis beta dari model eksekusi baru kini tersedia. Model eksekusi terarah memberikan fleksibilitas lebih saat mengustomisasi alur kerja mitigasi error kamu. Lihat panduan [Model eksekusi terarah](/guides/directed-execution-model) untuk informasi lebih lanjut.


<details>
<summary><b>Versi paket</b></summary>

Kode di halaman ini dikembangkan menggunakan persyaratan berikut.
Kami merekomendasikan menggunakan versi ini atau yang lebih baru.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
Teknik mitigasi error memungkinkan pengguna untuk memitigasi error pada Circuit dengan
memodelkan noise perangkat pada saat eksekusi. Ini biasanya
menghasilkan overhead pra-pemrosesan kuantum yang berkaitan dengan pelatihan model dan
overhead pasca-pemrosesan klasik untuk memitigasi error pada hasil mentah
menggunakan model yang dihasilkan.

Primitif Estimator mendukung beberapa teknik mitigasi error, termasuk [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation), [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne), [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec), dan [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options). Lihat [Teknik mitigasi dan supresi error](error-mitigation-and-suppression-techniques) untuk penjelasan masing-masing. Saat menggunakan primitif, kamu bisa mengaktifkan atau menonaktifkan metode individual. Lihat bagian [Pengaturan error kustom](#advanced-error) untuk detailnya.

> **Note:** Sampler tidak mendukung mitigasi error, tapi kamu bisa menggunakan paket [mthree](https://qiskit.github.io/qiskit-addon-mthree/) (matrix-free measurement mitigation) untuk melakukan mitigasi error secara lokal.

Estimator juga mendukung `resilience_level`. Level ketahanan menentukan seberapa besar ketahanan yang dibangun terhadap
error. Level yang lebih tinggi menghasilkan hasil yang lebih akurat, dengan trade-off
waktu pemrosesan yang lebih lama. Level ketahanan dapat digunakan untuk mengonfigurasi
trade-off biaya/akurasi saat menerapkan mitigasi error pada kueri primitif
kamu. Mitigasi error mengurangi error (bias) pada hasil dengan memproses
output dari kumpulan, atau ensemble, dari Circuit yang saling berkaitan. Tingkat
pengurangan error bergantung pada metode yang diterapkan. Level ketahanan mengabstraksikan
pilihan metode mitigasi error secara detail untuk memungkinkan
pengguna mempertimbangkan trade-off biaya/akurasi yang sesuai dengan
aplikasi mereka.

Dengan demikian, setiap level sesuai dengan satu atau beberapa metode dengan
overhead sampling kuantum yang semakin meningkat sehingga kamu bisa bereksperimen
dengan berbagai trade-off waktu-akurasi. Tabel berikut menunjukkan
level mana dan metode terkait yang tersedia untuk masing-masing
primitif.

> **Info:** Mitigasi error bersifat spesifik terhadap tugas, sehingga teknik yang bisa kamu
> terapkan bervariasi tergantung apakah kamu sedang mengambil sampel distribusi atau menghasilkan
> nilai ekspektasi.

<span id="resilience-table"></span>

Estimator mendukung level ketahanan berikut. Sampler tidak mendukung level ketahanan.

| Level Ketahanan | Definisi                                                                                              | Teknik                                                                    |
|-----------------|-------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0               | Tanpa mitigasi                                                                                        | Tidak ada                                                                 |
| 1 [Default]     | Biaya mitigasi minimal: Memitigasi error yang terkait dengan error readout                            | Twirled Readout Error eXtinction  (TREX) measurement twirling             |
| 2               | Biaya mitigasi sedang. Biasanya mengurangi bias pada estimator, tapi tidak dijamin bebas bias.        | Level 1 + Zero Noise Extrapolation (ZNE) dan gate twirling

> **Info:** Level ketahanan saat ini masih dalam beta sehingga overhead sampling dan
> kualitas solusi akan bervariasi dari satu Circuit ke Circuit lainnya. Fitur baru,
> opsi lanjutan, dan alat manajemen akan dirilis secara bertahap. Metode mitigasi error
> tertentu tidak dijamin akan diterapkan di setiap level ketahanan.

## Konfigurasi Estimator dengan level ketahanan
Kamu bisa menggunakan level ketahanan untuk menentukan teknik mitigasi error, atau kamu bisa mengatur teknik kustom secara individual seperti yang dijelaskan di [Pengaturan error kustom.](#advanced-error)

<details>
<summary>Resilience Level 0</summary>

Tidak ada mitigasi error yang diterapkan pada program pengguna.

</details>

<details>
<summary>Resilience Level 1</summary>

Level 1 menerapkan **mitigasi error readout** dan **measurement twirling** dengan menggunakan teknik bebas model yang dikenal
sebagai Twirled Readout Error eXtinction (TREX). Metode ini mengurangi error pengukuran
dengan mendiagonalisasi saluran noise yang terkait dengan pengukuran dengan
cara membalik Qubit secara acak melalui Gate X tepat sebelum pengukuran. Sebuah
suku penskalaan ulang dari saluran noise diagonal dipelajari dengan
membenchmark Circuit acak yang diinisialisasi dalam kondisi nol. Ini memungkinkan
layanan untuk menghilangkan bias dari nilai ekspektasi yang dihasilkan dari
noise readout. Pendekatan ini dijelaskan lebih lanjut di [Model-free
readout-error mitigation for quantum expectation
values](https://arxiv.org/abs/2012.09738).

</details>

<details>
<summary>Resilience Level 2</summary>

Level 2 menerapkan **teknik mitigasi error yang ada di level 1** dan juga menerapkan **gate twirling** serta menggunakan **metode Zero Noise Extrapolation (ZNE)**. ZNE menghitung
nilai ekspektasi dari observable untuk faktor noise yang berbeda
(tahap amplifikasi) dan kemudian menggunakan nilai ekspektasi yang diukur untuk
menyimpulkan nilai ekspektasi ideal pada batas zero-noise (tahap
ekstrapolasi). Pendekatan ini cenderung mengurangi error pada nilai ekspektasi, tapi
tidak dijamin menghasilkan hasil yang tidak bias.

![Gambar ini menunjukkan sebuah grafik. Sumbu-x diberi label Noise amplification factor. Sumbu-y diberi label Expectation value. Garis yang miring ke atas diberi label Mitigated value. Titik-titik di dekat garis adalah nilai yang diperkuat noise. Ada garis horizontal tepat di atas sumbu-X yang diberi label Exact value.](../docs/images/guides/configure-error-mitigation/resilience-2.svg "Ilustrasi metode ZNE")

Overhead dari metode ini berskala dengan jumlah faktor noise. Pengaturan
default mengambil sampel nilai ekspektasi pada tiga faktor noise,
menghasilkan overhead sekitar 3x saat menggunakan level ketahanan ini.

Di Level 2, metode TREX secara acak membalik Qubit melalui Gate X tepat sebelum pengukuran,
dan membalik bit yang diukur terkait jika Gate X diterapkan. Pendekatan ini dijelaskan lebih lanjut di [Model-free
readout-error mitigation for quantum expectation
values](https://arxiv.org/abs/2012.09738).

</details>

### Contoh
Antarmuka `EstimatorV2` memungkinkan pengguna bekerja secara mulus dengan berbagai
metode mitigasi error untuk mengurangi error pada nilai ekspektasi dari
observable. Kode berikut menggunakan Zero Noise Extrapolation dan mitigasi error readout dengan cukup
mengatur `resilience_level 2`.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## Pengaturan error kustom
Kamu bisa mengaktifkan dan menonaktifkan metode mitigasi dan supresi error secara individual, termasuk dynamical decoupling, gate dan measurement twirling, mitigasi error pengukuran, PEC, dan ZNE. Lihat [Teknik mitigasi dan supresi error](error-mitigation-and-suppression-techniques) untuk penjelasan masing-masing.

> **Note:** - Tidak semua opsi tersedia untuk kedua primitif. Lihat [tabel opsi yang tersedia](runtime-options-overview#options-table) untuk daftar opsi yang tersedia.
> - Tidak semua metode berfungsi bersama di semua jenis Circuit. Lihat [tabel kompatibilitas fitur](runtime-options-overview#options-compatibility-table) untuk detailnya.

<Tabs>
  <TabItem value="Estimator" label="Estimator">
    ```python
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}")
print(f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}")
```
  </TabItem>

  <TabItem value="Sampler" label="Sampler">